<a href="https://colab.research.google.com/github/FireDev2/FlorianWeinert-DataScience-GenAI-Submissions/blob/main/Assignment_5/6_02_DNN_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]]),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142., 155., 225.,  59

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 31267.7227
Epoch [10/100], Loss: 29913.5723
Epoch [10/100], Loss: 22801.4941
Epoch [10/100], Loss: 33198.3594
Epoch [10/100], Loss: 24481.7402
Epoch [10/100], Loss: 31005.1641
Epoch [10/100], Loss: 33931.2539
Epoch [10/100], Loss: 34343.125
Epoch [20/100], Loss: 32552.9785
Epoch [20/100], Loss: 25787.6504
Epoch [20/100], Loss: 41410.8359
Epoch [20/100], Loss: 26602.9062
Epoch [20/100], Loss: 21377.8594
Epoch [20/100], Loss: 27329.084
Epoch [20/100], Loss: 30216.2012
Epoch [20/100], Loss: 49827.293
Epoch [30/100], Loss: 25785.6035
Epoch [30/100], Loss: 33754.4258
Epoch [30/100], Loss: 27542.1816
Epoch [30/100], Loss: 29305.4922
Epoch [30/100], Loss: 33143.7188
Epoch [30/100], Loss: 28991.1445
Epoch [30/100], Loss: 27474.4688
Epoch [30/100], Loss: 28290.9961
Epoch [40/100], Loss: 28659.0039
Epoch [40/100], Loss: 35781.5742
Epoch [40/100], Loss: 31932.3223
Epoch [40/100], Loss: 27922.2051
Epoch [40/100], Loss: 24244.6016
Epoch [40/100], Loss: 27445.9746
Epoch [40/100

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 22024.28515625


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,15.789759,219.0
1,14.875921,70.0
2,15.641128,202.0
3,18.998446,230.0
4,15.099899,111.0
...,...,...
84,13.790284,153.0
85,12.644937,98.0
86,11.713119,37.0
87,12.054867,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

Excersice 1:

In [12]:
epochs = 1000 # Increased to 1000 epochs

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # update the parameters based on this round of training

  # every 100 steps we will print out the current loss for better readability with more epochs
  if (epoch+1) % 100 == 0: # modular arithmetic
      print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

print('\nTraining complete. Evaluating model...')

# Evaluation
model.eval() # testing mode
mse_values = [] # collect the MSE scores
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item())
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set after {epochs} epochs: {avg_mse}")

# Create DataFrame for comparison
results_df_1000_epochs = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
print('\nComparison of Predicted vs Actual values:')
print(results_df_1000_epochs.head())

Epoch [100/1000], Loss: 5502.9395
Epoch [200/1000], Loss: 3808.5542
Epoch [300/1000], Loss: 212.9713
Epoch [400/1000], Loss: 1333.4413
Epoch [500/1000], Loss: 802.0884
Epoch [600/1000], Loss: 416.8542
Epoch [700/1000], Loss: 672.0442
Epoch [800/1000], Loss: 307.7213
Epoch [900/1000], Loss: 1578.6208
Epoch [1000/1000], Loss: 2636.8396

Training complete. Evaluating model...
Average MSE on test set after 1000 epochs: 2851.297607421875

Comparison of Predicted vs Actual values:
    Predicted  Actual
0  141.422928   219.0
1  180.797775    70.0
2  139.295288   202.0
3  295.890350   230.0
4  121.846306   111.0


much more accurate

Excercise 2:

## Define an alternative model architecture

### Subtask:
Define a new PyTorch model class (`DiabetesModelV2`) with a modified architecture, specifically increasing the number of neurons in the two hidden layers from 5 to 10.


**Reasoning**:
To define the alternative model architecture as requested, I will create a new Python class `DiabetesModelV2` inheriting from `nn.Module` and set up its layers according to the instructions provided.



In [13]:
import torch.nn as nn

# Define the new model with increased neurons in hidden layers
class DiabetesModelV2(nn.Module):
    def __init__(self):
        super(DiabetesModelV2, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(10, 10), # First hidden layer: 10 input features, 10 neurons
            nn.ReLU(),
            nn.Linear(10, 10), # Second hidden layer: 10 input from previous, 10 neurons
            nn.ReLU(),
            nn.Linear(10, 1)   # Output layer: 10 input from previous, 1 output neuron
        )

    def forward(self, x):
        return self.layers(x)

print("DiabetesModelV2 class defined with updated architecture.")

DiabetesModelV2 class defined with updated architecture.


**Reasoning**:
Now that the `DiabetesModelV2` class is defined, the next step is to instantiate it, set up the loss function and optimizer, and train it for 1000 epochs using the provided `train_loader`, similar to how the previous model was trained.



In [14]:
import torch.optim as optim

# Instantiate the new model
model_v2 = DiabetesModelV2()

# Define loss function and optimizer for model_v2
criterion_v2 = nn.MSELoss()
optimiser_v2 = optim.Adam(model_v2.parameters(), lr=0.001)

epochs = 1000 # Use 1000 epochs as specified in the exercise

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_v2.to(device)

print("Starting training for DiabetesModelV2...")

for epoch in range(epochs):
  for inputs, targets in train_loader:
    inputs, targets = inputs.to(device), targets.to(device)

    model_v2.train() # Set the model to training mode
    optimiser_v2.zero_grad() # Reset the gradients
    outputs = model_v2(inputs) # Forward pass
    loss = criterion_v2(outputs, targets) # Calculate loss
    loss.backward() # Backward pass (backpropagation)
    optimiser_v2.step() # Update model parameters

  # Print loss every 100 epochs for readability
  if (epoch+1) % 100 == 0:
      print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}', flush=True)

print('\nTraining of DiabetesModelV2 complete.')

Starting training for DiabetesModelV2...
Epoch [100/1000], Loss: 20881.3984
Epoch [200/1000], Loss: 2171.9812
Epoch [300/1000], Loss: 4265.6343
Epoch [400/1000], Loss: 8305.1602
Epoch [500/1000], Loss: 2017.3684
Epoch [600/1000], Loss: 4048.1318
Epoch [700/1000], Loss: 287.6945
Epoch [800/1000], Loss: 674.0109
Epoch [900/1000], Loss: 844.1021
Epoch [1000/1000], Loss: 724.3972

Training of DiabetesModelV2 complete.


**Reasoning**:
Now that the `DiabetesModelV2` has been trained, the next logical step is to evaluate its performance on the test set. This involves calculating the average Mean Squared Error (MSE) and generating a DataFrame to compare the predicted values against the actual target values, similar to how the previous model was evaluated.



In [15]:
import numpy as np
import pandas as pd

# Evaluation for model_v2
model_v2.eval() # Set the model to evaluation mode
mse_values_v2 = [] # Collect MSE scores for model_v2
predictions_v2 = []
actuals_v2 = []

print("\nEvaluating DiabetesModelV2...")

with torch.no_grad(): # Disable gradient calculation during evaluation
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_v2(inputs) # Make predictions

        # Calculate Mean Squared Error for the batch
        mse = criterion_v2(outputs, targets)
        mse_values_v2.append(mse.item())

        # Store predictions and actuals
        predictions_v2.extend(outputs.cpu().numpy())
        actuals_v2.extend(targets.cpu().numpy())

# Calculate and print the average MSE for model_v2
avg_mse_v2 = np.mean(mse_values_v2)
print(f"Average MSE on test set for DiabetesModelV2 after {epochs} epochs: {avg_mse_v2}")

# Create DataFrame for comparison of Predicted vs Actual values for model_v2
results_df_v2 = pd.DataFrame({'Predicted': np.array(predictions_v2).flatten(), 'Actual': np.array(actuals_v2).flatten()})
print('\nComparison of Predicted vs Actual values for DiabetesModelV2:')
print(results_df_v2.head())



Evaluating DiabetesModelV2...
Average MSE on test set for DiabetesModelV2 after 1000 epochs: 2845.7353515625

Comparison of Predicted vs Actual values for DiabetesModelV2:
    Predicted  Actual
0  146.364151   219.0
1  180.594070    70.0
2  142.410904   202.0
3  299.161987   230.0
4  126.716019   111.0


### Comparison of DiabetesModelV2 (10 neurons) vs. DiabetesModel (5 neurons) after 1000 Epochs

Let's compare the performance of the `DiabetesModelV2` (with 10 neurons in hidden layers) against the `DiabetesModel` (with 5 neurons in hidden layers) after both were trained for 1000 epochs.

**Average MSE on Test Set:**
- `DiabetesModel` (original architecture, 1000 epochs): **`avg_mse`**
- `DiabetesModelV2` (10 neurons, 1000 epochs): **`avg_mse_v2`**

**Comparison of Predicted vs. Actual values (First 5 rows):**

**`DiabetesModel` (Original Architecture, 1000 Epochs):**
```
`results_df_1000_epochs.head().to_markdown(index=False)`
```

**`DiabetesModelV2` (10 Neurons, 1000 Epochs):**
```
`results_df_v2.head().to_markdown(index=False)`
```

From the comparison:
*   The `avg_mse_v2` for `DiabetesModelV2` (approximately `avg_mse_v2`) is slightly lower than `avg_mse` for the original `DiabetesModel` (approximately `avg_mse`). This suggests a marginal improvement in performance with the increased number of neurons in the hidden layers.
*   Visually inspecting the predicted vs. actual values also shows that both models are providing predictions that are closer to the actual values compared to the initial 100-epoch run. `DiabetesModelV2` predictions, while similar to `DiabetesModel`, might show minor differences reflecting its slightly better MSE.

**Suggestions for further architectural experiments:**
1.  **More Layers:** Try adding more hidden layers to the network (e.g., 3 or 4 hidden layers). Deeper networks can sometimes capture more complex patterns.
2.  **More Neurons:** Experiment with even more neurons per layer (e.g., 20, 50, or 100) to see if the model can learn more intricate relationships. Be cautious of overfitting with too many parameters.
3.  **Different Activation Functions:** Explore activation functions other than ReLU, such as Leaky ReLU, ELU, or Tanh, especially in deeper networks.
4.  **Regularization:** Implement regularization techniques like Dropout in the hidden layers to prevent overfitting, which might become more prominent with larger models.
5.  **Batch Normalization:** Add Batch Normalization layers between linear layers and activation functions to stabilize training and potentially speed up convergence.
6.  **Learning Rate Scheduling:** Experiment with different learning rate schedules (e.g., learning rate decay) in the optimizer to fine-tune the training process.
7.  **Different Optimizers:** While Adam is a good general-purpose optimizer, trying others like SGD with momentum or RMSprop might yield different results.

These experiments can help identify an optimal architecture that balances model complexity with performance and generalization on unseen data.

### Comparison of DiabetesModelV2 (10 neurons) vs. DiabetesModel (5 neurons) after 1000 Epochs

Let's compare the performance of the `DiabetesModelV2` (with 10 neurons in hidden layers) against the `DiabetesModel` (with 5 neurons in hidden layers) after both were trained for 1000 epochs.

**Average MSE on Test Set:**
- `DiabetesModel` (original architecture, 1000 epochs): **2851.2976**
- `DiabetesModelV2` (10 neurons, 1000 epochs): **2845.7354**

**Comparison of Predicted vs. Actual values (First 5 rows):**

**`DiabetesModel` (Original Architecture, 1000 Epochs):**
```
| Predicted | Actual |
|:----------|:-------|
| 141.422928 | 219.0  |
| 180.797775 | 70.0   |
| 139.295288 | 202.0  |
| 295.890350 | 230.0  |
| 121.846306 | 111.0  |
```

**`DiabetesModelV2` (10 Neurons, 1000 Epochs):**
```
| Predicted | Actual |
|:----------|:-------|
| 146.364151 | 219.0  |
| 180.594070 | 70.0   |
| 142.410904 | 202.0  |
| 299.161987 | 230.0  |
| 126.716019 | 111.0  |
```

From the comparison:
*   The `avg_mse_v2` for `DiabetesModelV2` (approximately 2845.7354) is slightly lower than `avg_mse` for the original `DiabetesModel` (approximately 2851.2976). This suggests a marginal improvement in performance with the increased number of neurons in the hidden layers.
*   Visually inspecting the predicted vs. actual values also shows that both models are providing predictions that are closer to the actual values compared to the initial 100-epoch run. `DiabetesModelV2` predictions, while similar to `DiabetesModel`, might show minor differences reflecting its slightly better MSE.

**Suggestions for further architectural experiments:**
1.  **More Layers:** Try adding more hidden layers to the network (e.g., 3 or 4 hidden layers). Deeper networks can sometimes capture more complex patterns.
2.  **More Neurons:** Experiment with even more neurons per layer (e.g., 20, 50, or 100) to see if the model can learn more intricate relationships. Be cautious of overfitting with too many parameters.
3.  **Different Activation Functions:** Explore activation functions other than ReLU, such as Leaky ReLU, ELU, or Tanh, especially in deeper networks.
4.  **Regularization:** Implement regularization techniques like Dropout in the hidden layers to prevent overfitting, which might become more prominent with larger models.
5.  **Batch Normalization:** Add Batch Normalization layers between linear layers and activation functions to stabilize training and potentially speed up convergence.
6.  **Learning Rate Scheduling:** Experiment with different learning rate schedules (e.g., learning rate decay) in the optimizer to fine-tune the training process.
7.  **Different Optimizers:** While Adam is a good general-purpose optimizer, trying others like SGD with momentum or RMSprop might yield different results.

These experiments can help identify an optimal architecture that balances model complexity with performance and generalization on unseen data.

### Comparison of DiabetesModelV2 (10 neurons) vs. DiabetesModel (5 neurons) after 1000 Epochs

Let's compare the performance of the `DiabetesModelV2` (with 10 neurons in hidden layers) against the `DiabetesModel` (with 5 neurons in hidden layers) after both were trained for 1000 epochs.

**Average MSE on Test Set:**
- `DiabetesModel` (original architecture, 1000 epochs): **2851.2976**
- `DiabetesModelV2` (10 neurons, 1000 epochs): **2845.7354**

**Comparison of Predicted vs. Actual values (First 5 rows):**

**`DiabetesModel` (Original Architecture, 1000 Epochs):**
```
| Predicted | Actual |
|:----------|:-------|
| 141.422928 | 219.0  |
| 180.797775 | 70.0   |
| 139.295288 | 202.0  |
| 295.890350 | 230.0  |
| 121.846306 | 111.0  |
```

**`DiabetesModelV2` (10 Neurons, 1000 Epochs):**
```
| Predicted | Actual |
|:----------|:-------|
| 146.364151 | 219.0  |
| 180.594070 | 70.0   |
| 142.410904 | 202.0  |
| 299.161987 | 230.0  |
| 126.716019 | 111.0  |
```

From the comparison:
*   The `avg_mse_v2` for `DiabetesModelV2` (approximately 2845.7354) is slightly lower than `avg_mse` for the original `DiabetesModel` (approximately 2851.2976). This suggests a marginal improvement in performance with the increased number of neurons in the hidden layers.
*   Visually inspecting the predicted vs. actual values also shows that both models are providing predictions that are closer to the actual values compared to the initial 100-epoch run. `DiabetesModelV2` predictions, while similar to `DiabetesModel`, might show minor differences reflecting its slightly better MSE.

**Suggestions for further architectural experiments:**
1.  **More Layers:** Try adding more hidden layers to the network (e.g., 3 or 4 hidden layers). Deeper networks can sometimes capture more complex patterns.
2.  **More Neurons:** Experiment with even more neurons per layer (e.g., 20, 50, or 100) to see if the model can learn more intricate relationships. Be cautious of overfitting with too many parameters.
3.  **Different Activation Functions:** Explore activation functions other than ReLU, such as Leaky ReLU, ELU, or Tanh, especially in deeper networks.
4.  **Regularization:** Implement regularization techniques like Dropout in the hidden layers to prevent overfitting, which might become more prominent with larger models.
5.  **Batch Normalization:** Add Batch Normalization layers between linear layers and activation functions to stabilize training and potentially speed up convergence.
6.  **Learning Rate Scheduling:** Experiment with different learning rate schedules (e.g., learning rate decay) in the optimizer to fine-tune the training process.
7.  **Different Optimizers:** While Adam is a good general-purpose optimizer, trying others like SGD with momentum or RMSprop might yield different results.

These experiments can help identify an optimal architecture that balances model complexity with performance and generalization on unseen data.


## Compare results and suggest next steps

### Subtask:
Analyze the average MSE and the predicted vs. actual values from the new architecture. Compare these results with the previous run (1000 epochs with the original architecture) and discuss whether the new architecture is better and what further architectural experiments could be performed to search for an optimal configuration.


## Summary:

### Q&A
The new `DiabetesModelV2` architecture, with 10 neurons in its hidden layers, is marginally better than the original `DiabetesModel` (with 5 neurons) when both are trained for 1000 epochs. This is indicated by its slightly lower average Mean Squared Error (MSE) on the test set.

### Data Analysis Key Findings
*   A new model, `DiabetesModelV2`, was defined with two hidden layers, each containing 10 neurons, an increase from the original model's 5 neurons per hidden layer.
*   After training `DiabetesModelV2` for 1000 epochs, its average MSE on the test set was calculated as 2845.7354.
*   Comparing this to the original `DiabetesModel`'s average MSE of 2851.2976 (after 1000 epochs), `DiabetesModelV2` achieved a marginal improvement with a slightly lower MSE.
*   Both models showed predictions that were closer to actual values compared to earlier runs with fewer epochs, with `DiabetesModelV2` exhibiting minor differences consistent with its better MSE.

### Insights or Next Steps
*   Further architectural experiments should be conducted, including increasing the number of hidden layers or neurons, exploring different activation functions (e.g., Leaky ReLU, Tanh), and implementing regularization techniques like Dropout to prevent overfitting.
*   Investigate advanced optimization strategies such as Batch Normalization, learning rate scheduling, or alternative optimizers (e.g., SGD with momentum, RMSprop) to potentially enhance model performance and training stability.
